# Minimum Wages and Employment: A Case Study of the Fast-Food Industry in New Jersey and Pennsylvania

本工作簿复现 David Card 与 Alan B. Krueger 的经典论文 *Minimum Wages and Employment: A Case Study of the Fast-Food Industry in New Jersey and Pennsylvania*（AER, 1994）中的：

- **表 2** — 各变量均值（按州，分第一波/第二波）
- **表 3** — 每家门店平均就业人数（最低工资上调前后 + 双重差分 DiD）
- **表 4** — 就业变化的简化式（reduced-form）回归
- **图 1** — 起始工资（starting wage）分布，新泽西 vs 宾夕法尼亚，上调前 vs 上调后

**研究背景**：1992 年 4 月，新泽西州把最低工资从 \$4.25/小时提高到 \$5.05/小时；邻近的宾夕法尼亚州不变，作为对照组。作者对 410 家快餐店做了两轮调查（第一波 1992 年 2 月，第二波 1992 年 11–12 月），检验最低工资上调是否减少了就业。

**数据**：`public.csv`（与 notebook 同目录），410 个观测、46 列。空白表示缺失。

从上到下依次运行每个单元格即可。每节标注了 **输入 → 操作 → 输出**。


## 0. 读入数据与准备

**输入**：`public.csv`。**操作**：读入 CSV，把空白当作缺失值（NaN），全部转为数值。**输出**：`df`（410 行 × 46 列）。

关键变量（详见 DATA_DICTIONARY）：
- `STATE`：1 = 新泽西(NJ)，0 = 宾夕法尼亚(PA)
- `EMPFT`/`EMPPT`/`NMGRS`：第一波 全职/兼职/经理人数；`EMPFT2`/`EMPPT2`/`NMGRS2` 为第二波
- `WAGE_ST`/`WAGE_ST2`：起始工资（美元/小时）
- `STATUS2`：第二波状态（0 拒绝、1 完成、2/4/5 临时关闭、3 永久关闭）
- `CHAIN`：1=BK、2=KFC、3=Roy Rogers、4=Wendy's


In [ ]:
# ===== SETUP：读入数据 =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from IPython.display import display

DATA = "public.csv"   # 如路径不同，请修改这里
df = pd.read_csv(DATA)

# 空白字符串 -> NaN，再统一转成数值
df = df.replace(r"^\s*$", np.nan, regex=True)
for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("观测数(门店):", len(df), "| 列数:", df.shape[1])
print("按州分布:")
display(df["STATE"].value_counts().rename({1: "新泽西 NJ", 0: "宾夕法尼亚 PA"}).to_frame("门店数"))


## 1. 构造变量

**输入**：`df`。**操作**：
- **全职当量就业** FTE = 全职 + 经理 + 0.5 × 兼职（兼职按半个全职计）
- 永久关闭的门店（`STATUS2==3`）第二波就业设为 0；临时关闭（`STATUS2∈{2,4,5}`）保持缺失
- `demp` = 第二波 FTE − 第一波 FTE（就业变化）
- `gap` = 为达到新最低工资所需的工资涨幅（初始工资缺口），PA 或初始工资≥5.05 时为 0
- 其它：全职占比、套餐价格、按州×工资档的分组

**输出**：`df` 新增若干列。


In [ ]:
# ===== 1a. 全职当量就业（FTE）=====
# 兼职按 0.5 个全职计；经理计入全职
df["fte1"] = df["EMPFT"] + df["NMGRS"] + 0.5 * df["EMPPT"]     # 第一波
df["fte2"] = df["EMPFT2"] + df["NMGRS2"] + 0.5 * df["EMPPT2"]   # 第二波
# 永久关闭的门店(STATUS2==3)第二波就业在数据中已是 0；临时关闭(2,4,5)保持缺失
df["demp"] = df["fte2"] - df["fte1"]      # 就业变化

# ===== 1b. 初始工资缺口 gap =====
# gap = 0（PA 或 NJ 初始工资 >= 5.05）；否则 = (5.05 - wage_st)/wage_st
df["gap"] = np.nan
df.loc[df["STATE"] == 0, "gap"] = 0.0
nj = df["STATE"] == 1
df.loc[nj & (df["WAGE_ST"] >= 5.05), "gap"] = 0.0
df.loc[nj & (df["WAGE_ST"] > 0) & (df["WAGE_ST"] < 5.05), "gap"] = (5.05 - df["WAGE_ST"]) / df["WAGE_ST"]

# ===== 1c. 其它派生变量 =====
df["fracft"] = df["EMPFT"] / df["fte1"]      # 全职占比（按 FTE 口径）
df["fracft2"] = df["EMPFT2"] / df["fte2"]
df["pmeal"] = df["PSODA"] + df["PFRY"] + df["PENTREE"]    # 套餐价格 = 汽水+薯条+主菜
df["pmeal2"] = df["PSODA2"] + df["PFRY2"] + df["PENTREE2"]

# ===== 1d. 按 州×初始工资档 分组（用于表3）=====
def wage_group(r):
    if r["STATE"] == 0:
        return "PA"
    if r["WAGE_ST"] == 4.25:
        return "NJ_low"
    if 4.25 < r["WAGE_ST"] < 5.00:
        return "NJ_mid"
    if r["WAGE_ST"] >= 5.00:
        return "NJ_high"
    return "NJ_missing"
df["group"] = df.apply(wage_group, axis=1)

df["chain_name"] = df["CHAIN"].map({1: "Burger King", 2: "KFC", 3: "Roy Rogers", 4: "Wendy's"})
print("FTE 第一波均值(全体): %.3f | 第二波均值(全体): %.3f" % (df["fte1"].mean(), df["fte2"].mean()))
print("就业变化 demp: 均值 %.3f | 标准差 %.3f" % (df["demp"].mean(), df["demp"].std(ddof=1)))


## 2. 表 2 — 各变量均值（按州）

**输入**：`df`。**操作**：分 NJ / PA 计算各变量的均值（对非缺失观测），给出标准误（= 标准差/√n），以及 NJ−PA 均值相等性检验的 t 统计量。**输出**：表 2。

说明：论文中的"百分比全职"= 全职人数 / 全职当量（`EMPFT/FTE`），套餐价格 = 汽水+薯条+主菜三样之和，第二波的"招募奖金"用 `SPECIAL2`。


In [ ]:
# ===== 2a. 门店类型分布（百分比，按州）=====
chain_pct = (df.groupby(["STATE", "chain_name"]).size().unstack(fill_value=0))
chain_pct = chain_pct.div(chain_pct.sum(axis=1), axis=0) * 100
chain_pct.index = ["宾夕法尼亚 PA", "新泽西 NJ"]
co_owned = (df.groupby("STATE")["CO_OWNED"].mean() * 100).reindex([0, 1]).values
chain_pct["公司直营占比(%)"] = co_owned
print("门店类型分布（%）与公司直营占比：")
display(chain_pct.round(1))


In [ ]:
# ===== 2b. 均值 + 标准误 + t 统计量 =====
def mean_se_t(varname):
    """返回 (NJ均值, NJ标准误, PA均值, PA标准误, t统计量)"""
    v_nj = df.loc[df.STATE == 1, varname].dropna()
    v_pa = df.loc[df.STATE == 0, varname].dropna()
    se_nj = v_nj.std(ddof=1) / np.sqrt(len(v_nj))
    se_pa = v_pa.std(ddof=1) / np.sqrt(len(v_pa))
    t = (v_nj.mean() - v_pa.mean()) / np.sqrt(se_nj**2 + se_pa**2)
    return v_nj.mean(), se_nj, v_pa.mean(), se_pa, t

def pct_row(varname):
    """百分比型变量：均值和标准误×100，t 统计量保持不变"""
    m_nj, se_nj, m_pa, se_pa, t = mean_se_t(varname)
    return 100*m_nj, 100*se_nj, 100*m_pa, 100*se_pa, t

# 需要先构造指示变量（0/1），用于计算百分比型变量
df["at425"] = (df["WAGE_ST"] == 4.25).astype(float)
df["at425_2"] = (df["WAGE_ST2"] == 4.25).astype(float)
df["at505_2"] = (df["WAGE_ST2"] == 5.05).astype(float)

rows = {
    "第1波: FTE 就业": mean_se_t("fte1"),
    "第1波: 全职占比(%)": pct_row("fracft"),
    "第1波: 起始工资": mean_se_t("WAGE_ST"),
    "第1波: 工资=$4.25(%)": pct_row("at425"),
    "第1波: 套餐价格": mean_se_t("pmeal"),
    "第1波: 营业小时": mean_se_t("HRSOPEN"),
    "第1波: 招募奖金(%)": pct_row("BONUS"),
    "第2波: FTE 就业": mean_se_t("fte2"),
    "第2波: 全职占比(%)": pct_row("fracft2"),
    "第2波: 起始工资": mean_se_t("WAGE_ST2"),
    "第2波: 工资=$4.25(%)": pct_row("at425_2"),
    "第2波: 工资=$5.05(%)": pct_row("at505_2"),
    "第2波: 套餐价格": mean_se_t("pmeal2"),
    "第2波: 营业小时": mean_se_t("HRSOPEN2"),
    "第2波: 招募奖金(%)": pct_row("SPECIAL2"),
}

tab2 = pd.DataFrame(rows, index=["新泽西 NJ", "标准误", "宾夕法尼亚 PA", "标准误", "t 统计量"]).T
tab2.columns = ["NJ 均值", "NJ 标准误", "PA 均值", "PA 标准误", "t 统计量"]
display(tab2.round(2))


## 3. 图 1 — 起始工资分布

**输入**：`df` 的 `WAGE_ST`（第一波）与 `WAGE_ST2`（第二波）。**操作**：画 2×2 的直方图（上行 = 第一波 NJ/PA，下行 = 第二波 NJ/PA）。**输出**：图 1。

核心发现：第一波两州起始工资分布几乎重合；第二波新泽西的分布整体右移、集中在 \$5.05，而宾夕法尼亚基本不变。


In [ ]:
# ===== 图1：起始工资分布（2×2，标签用英文以匹配论文原图）=====
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
bins = np.arange(4.0, 6.5, 0.25)

for col, st in [(0, 1), (1, 0)]:          # 列：NJ(左) / PA(右)
    lab = "New Jersey" if st == 1 else "Pennsylvania"
    for row, (var, wave) in enumerate([("WAGE_ST", "February 1992"),
                                       ("WAGE_ST2", "November 1992")]):
        ax = axes[row, col]
        s = df.loc[df.STATE == st, var].dropna()
        ax.hist(s, bins=bins, color=("tab:blue" if st == 1 else "tab:orange"),
                alpha=0.75, edgecolor="white")
        ax.axvline(4.25, color="gray", linewidth=0.8, linestyle="--")
        ax.axvline(5.05, color="gray", linewidth=0.8, linestyle="--")
        if row == 0:
            ax.set_title(lab, fontsize=12)
        if col == 0:
            ax.set_ylabel(wave + "\nNumber of stores", fontsize=10)

axes[1, 0].set_xlabel("Starting wage ($/hour)")
axes[1, 1].set_xlabel("Starting wage ($/hour)")
fig.suptitle("Figure 1  Distribution of starting wage rates (dashed: $4.25 and $5.05)", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## 4. 表 3 — 平均就业与双重差分（DiD）

**输入**：`df`。**操作**：分 5 个组（PA、NJ、NJ 低薪/中档/高薪）计算：

- 行 1：第一波 FTE 均值（全部可用观测）
- 行 2：第二波 FTE 均值（全部可用观测，永久关闭店 = 0）
- 行 3：平均 FTE 的变化 = 行 2 − 行 1（非平衡）
- 行 4：变化（平衡样本，即两波都有数据的门店）
- 行 5：变化（临时关闭店第二波就业设为 0）

并计算差异列：NJ−PA（DiD）、低薪−高薪、中档−高薪。**输出**：表 3。

> 注：行 3 的"变化"= 两波均值之差（非平衡），其标准误按配对样本（同一批门店两波之差）计算；论文该行报告的标准误（NJ 0.54、PA 1.25、DiD 1.36）与本表的配对标准误（NJ 0.48、PA 1.25、DiD 1.34）略有出入，但由此得到的 **DiD t 统计量 ≈ 2.03，与论文一致**。行 1、2、4、5 与论文完全一致。


In [ ]:
# ===== 表3 计算 =====
groups = {"PA": ("宾夕法尼亚 PA", df.STATE == 0),
          "NJ": ("新泽西 NJ", df.STATE == 1),
          "NJ_low": ("NJ 低薪 $4.25", df.group == "NJ_low"),
          "NJ_mid": ("NJ 中档 $4.26-4.99", df.group == "NJ_mid"),
          "NJ_high": ("NJ 高薪 ≥$5.00", df.group == "NJ_high")}

def rstat(mask):
    """对某个组返回 行1~行5 的 (mean, se)"""
    g = df.loc[mask]
    m1, se1 = g["fte1"].mean(), g["fte1"].std(ddof=1) / np.sqrt(g["fte1"].count())
    m2, se2 = g["fte2"].mean(), g["fte2"].std(ddof=1) / np.sqrt(g["fte2"].count())
    # 行4: 平衡样本（两波都有数据的门店）——这是 DiD 的标准样本
    bal = g.loc[g.fte1.notna() & g.fte2.notna()]
    d4 = (bal["fte2"] - bal["fte1"])
    m4, se4 = d4.mean(), d4.std(ddof=1) / np.sqrt(len(d4))
    # 行3: 非平衡两波均值之差，标准误用配对样本（同一批门店两波之差）
    chg3 = m2 - m1
    se3 = se4
    # 行5: 临时关闭店第二波=0（只对 STATUS2∈{2,4,5} 设0，其余缺失保持缺失）
    base5 = g.loc[g.fte1.notna() & (g.fte2.notna() | g.STATUS2.isin([2, 4, 5]))].copy()
    base5["fte2_5"] = base5["fte2"].where(base5["fte2"].notna(), 0.0)
    d5 = base5["fte2_5"] - base5["fte1"]
    m5, se5 = d5.mean(), d5.std(ddof=1) / np.sqrt(len(d5))
    return [(m1, se1), (m2, se2), (chg3, se3), (m4, se4), (m5, se5)]

res = {k: rstat(v[1]) for k, v in groups.items()}

def fmt(pair):
    return f"{pair[0]:.2f} ({pair[1]:.2f})"

def diff(a, b):
    """两列之差及其标准误 = sqrt(se_a^2+se_b^2)"""
    return (a[0] - b[0], np.sqrt(a[1]**2 + b[1]**2))

rows_names = ["1. 第一波 FTE 就业（全部可用）",
              "2. 第二波 FTE 就业（全部可用）",
              "3. 平均 FTE 变化（非平衡）",
              "4. 平均 FTE 变化（平衡样本）",
              "5. 平均 FTE 变化（临时关闭店=0）"]

tab3 = pd.DataFrame(index=rows_names)
for col, key in [("PA", "PA"), ("NJ", "NJ"), ("NJ−PA", None),
                 ("NJ 低薪$4.25", "NJ_low"), ("NJ 中档", "NJ_mid"),
                 ("NJ 高薪≥$5.00", "NJ_high"),
                 ("低薪−高薪", None), ("中档−高薪", None)]:
    if key is not None:
        vals = res[key]
    elif col == "NJ−PA":
        vals = [diff(res["NJ"][i], res["PA"][i]) for i in range(5)]
    elif col == "低薪−高薪":
        vals = [diff(res["NJ_low"][i], res["NJ_high"][i]) for i in range(5)]
    else:  # 中档−高薪
        vals = [diff(res["NJ_mid"][i], res["NJ_high"][i]) for i in range(5)]
    tab3[col] = [fmt(v) for v in vals]

tab3 = tab3.T
print("表 3：每家门店平均就业人数（单元格 = 均值 (标准误)）")
display(tab3)


## 5. 表 4 — 就业变化的简化式回归

**输入**：`df` 中两波就业与工资都有效（或永久关闭）的门店（357 家）。**操作**：以就业变化 `demp` 为被解释变量，估计 5 个 OLS 模型：

| 列 | 设定 |
|---|---|
| (i) | NJ 虚拟变量 |
| (ii) | NJ 虚拟变量 + 连锁品牌与直营控制 |
| (iii) | 工资缺口 GAP |
| (iv) | GAP + 连锁品牌与直营控制 |
| (v) | GAP + 连锁品牌 + 地区虚拟变量 |

报告系数、标准误、回归标准误（√MSE）、控制变量联合 F 检验 p 值。**输出**：表 4。


In [ ]:
# ===== 表4 样本与回归 =====
df["dwage"] = df["WAGE_ST2"] - df["WAGE_ST"]
sub = df[df["demp"].notna() & ((df["STATUS2"] == 3) | df["dwage"].notna())].copy()
print("表4 样本量:", len(sub), "（论文为 357）")
print("被解释变量 demp 的均值: %.3f | 标准差: %.3f" % (sub["demp"].mean(), sub["demp"].std(ddof=1)))

sub["bk"] = (sub["CHAIN"] == 1).astype(float)
sub["kfc"] = (sub["CHAIN"] == 2).astype(float)
sub["roys"] = (sub["CHAIN"] == 3).astype(float)

specs = [
    ("(i)   NJ 虚拟变量", ["STATE"], "STATE"),
    ("(ii)  NJ + 连锁/直营", ["STATE", "bk", "kfc", "roys", "CO_OWNED"], "STATE"),
    ("(iii) GAP", ["gap"], "gap"),
    ("(iv)  GAP + 连锁/直营", ["gap", "bk", "kfc", "roys", "CO_OWNED"], "gap"),
    ("(v)   GAP + 连锁 + 地区", ["gap", "bk", "kfc", "roys", "CENTRALJ", "SOUTHJ", "PA1", "PA2"], "gap"),
]

rows4 = []
for name, xvars, key in specs:
    X = sm.add_constant(sub[xvars])
    fit = sm.OLS(sub["demp"], X).fit()
    coef, se = fit.params[key], fit.bse[key]
    # 控制变量联合 F 检验（排除关键变量与常数）
    ctrl = [x for x in xvars if x != key]
    if ctrl:
        Xr = sm.add_constant(sub[[key]])
        fitr = sm.OLS(sub["demp"], Xr).fit()
        rss_f, rss_r = np.sum(fit.resid**2), np.sum(fitr.resid**2)
        df1, df2 = len(ctrl), int(fit.df_resid)
        F = ((rss_r - rss_f) / df1) / (rss_f / df2)
        p = 1 - stats.f.cdf(F, df1, df2)
    else:
        p = np.nan
    rows4.append({"模型": name, "系数": coef, "标准误": se,
                  "回归标准误 √MSE": np.sqrt(fit.mse_resid),
                  "样本量": int(fit.nobs), "控制变量 p 值": p})

tab4 = pd.DataFrame(rows4).set_index("模型")
display(tab4.round(4))


## 6. 结果核对与解读

跑完上述单元格后，把本表的数字与论文原文（表 2~4、图 1）对照。关键结果应吻合：

- **表 3**：NJ 第一波 FTE ≈ 20.44、第二波 ≈ 21.03；PA 第一波 ≈ 23.33、第二波 ≈ 21.17；DiD ≈ 2.75（论文正文记 2.76，t≈2.03）。
- **表 4**：NJ 虚拟变量系数 ≈ 2.33 (1.19)；GAP 系数 ≈ 15.65 (6.08)，加入控制后 ≈ 14.92 (6.21)、加入地区后 ≈ 11.91 (7.39)。
- **图 1**：第一波两州工资分布相近，第二波 NJ 明显右移并集中到 \$5.05。

**结论解读**：与"竞争性劳动市场"的预测相反，最低工资上调**没有**导致新泽西快餐业就业下降（相对 PA 甚至略有上升）。这一结论在后续文献中引发大量讨论（如 Neumark & Wascher 2000 的质疑与 Card & Krueger 2000 的回应）。
